# 05 · Membership-inference attacks

**Purpose.** Run all attacks on every target model and checkpoint: A1 Shokri (Fisher LDA), A1b per-class Shokri, A2 loss, A3 confidence, A4 modified entropy, A5 K-means and A6 Fuzzy C-means.

**Inputs.** Target and shadow probability caches (notebooks 02–04).

**Outputs.** `attack_runs` (AUC with bootstrap CI, advantage, TPR at 1% and 0.1% FPR, lift, severity) and 20,000 `sample_scores` rows per run.

**Syllabus topics.** LDA (supervised attack); K-means and Fuzzy C-means clustering (unsupervised attacks).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import time

from mia import db
from mia.attacks import run_all_attacks
from mia.training import Workspace

ws = Workspace.open(need_features=False)
targets = db.models_df(ws.conn, role="target")["model_id"].tolist()
failed, kmeans_points = [], []
for i, mid in enumerate(targets, 1):
    t0 = time.perf_counter()
    try:
        res = run_all_attacks(ws.conn, ws.exp, mid, ws.labels)
        kmeans_points += [(mid, r["tpr"], r["fpr"]) for r in res if r["attack"] == "kmeans"]
        print(f"[{i:>3}/{len(targets)}] {mid:<26} {len(res)} new attack run(s) ({time.perf_counter() - t0:.1f} s)")
    except Exception as e:  # keep going; report at the end
        failed.append(mid)
        print(f"[{i:>3}/{len(targets)}] {mid:<26} FAILED: {type(e).__name__}: {e}")
print("failed targets:", failed or "none")

## AUC [95% CI] per target and attack
Every table also reports TPR at 1% and 0.1% FPR, because AUC alone hides the low-FPR regime that matters to an attacker.

In [ ]:
import pandas as pd

runs = db.attack_runs_df(ws.conn)
runs["AUC [CI]"] = runs.apply(lambda r: "—" if pd.isna(r["auc"]) else f"{r['auc']:.3f} [{r['auc_ci_low']:.3f}, {r['auc_ci_high']:.3f}]", axis=1)
runs.pivot(index="target_model_id", columns="attack", values="AUC [CI]")

In [ ]:
runs.pivot(index="target_model_id", columns="attack", values="tpr_at_1pct_fpr").round(4)

In [ ]:
runs.pivot(index="target_model_id", columns="attack", values="tpr_at_01pct_fpr").round(4)

In [ ]:
if kmeans_points:
    print(pd.DataFrame(kmeans_points, columns=["model_id", "kmeans TPR", "kmeans FPR"]).round(4).to_string(index=False))
best = db.leakage_vs_gap_df(ws.conn).drop_duplicates("model_id")
print(best[["model_id", "attack", "auc", "tpr_at_01pct_fpr", "lift_at_01pct_fpr", "severity"]].round(4).to_string(index=False))

## Results
The final printout lists the strongest attack per target with its lift and severity; the pivots give the full attack × model picture. K-means yields a single operating point, so it has TPR/FPR instead of an AUC.